# Lesson 4, Part 3

In [ ]:
%pip install ipycc numpy
from ipycc.turtle import Turtle, showscreen, clearscreen, bgcolor
import numpy as np

Consider a small object moving counterclockwise with constant speed *v* in a circle of radius *r*. 
Horizontal velocity is -*yv/r* while vertical velocity is *xv/r*.
Let's track motion assuming these are constant over short instants *dt* and display the results using turtle.

In [ ]:
def circular_motion(radius,velocity,num=1000,circuits=3):
    circumference = 2 * np.pi * radius
    period =  circumference / velocity
    agg_time = circuits * period 
    time = np.linspace(0,agg_time,num+1)
    x,y = np.zeros(num+1),np.zeros(num+1)
    ratio_vr = velocity / radius
    dt = agg_time / num
    x[0] = radius
    for i in range(num):
        vx = -y[i] * ratio_vr      # velocity in x direction
        vy = x[i] * ratio_vr       # velocity in y direction
        x[i+1] = x[i] + vx * dt
        y[i+1] = y[i] + vy * dt
    return time,x,y
 
def plot_xy(x,y,pen_color='black'):
    t.color(pen_color)
    t.penup(); t.goto(x[0],y[0]); t.pendown()
    for i in range(1,len(x)):
        t.goto(x[i],y[i])    
    
showcscreen()
clearscreen()
radius,velocity = 150,10
time_values,x,y = circular_motion(radius,velocity,num=500)
t.shape('circle')
t.speed('fastest')
plot_xy(x,y,'blue')

Since each tangent extends beyond the starting radius, the orbit slowly spirals outward. 
The overshoot is roughly proportional to *dt*.

In [ ]:
showscreen()
clearscreen()
radius,velocity = 150,10
for num in [500,1000,10000,100000]:
    time_values,x,y = circular_motion(radius,velocity,num)
    end_radius = np.sqrt(x[-1]**2 + y[-1]**2)
    print('num =',num,', last radius / first radius =',round(end_radius/radius,5))

For better approximation with less computation, 
we can estimate the velocity between (*x*[i],*y*[i]) and (*x*[i+1],*y*[i+1]) 
as a near-average value over the interval.

In [ ]:
def circular_motion2(radius,velocity,num=500,circuits=3):
    period = 2 * np.pi * radius / velocity
    agg_time =  circuits * period
    time = np.linspace(0,agg_time,num+1)
    x,y = np.zeros(num+1),np.zeros(num+1)
    ratio_vr = velocity / radius
    dt = agg_time / num
    x[0] = curr_x = radius
    curr_y = 0    
    for i in range(1,num+1):
        vx = -curr_y * ratio_vr
        vy = curr_x * ratio_vr  
        avg_x = curr_x + 0.5 * vx * dt  # average x between i and i+1 if velocity vx constant
        avg_y = curr_y + 0.5 * vy * dt  # average y between i and i+1 if velocity vy constant
        avg_vx = -avg_y * ratio_vr      # vx recalculated from avg_y instead of y[i]
        avg_vy = avg_x * ratio_vr       # vy recalculated from avg_x instead of x[i]
        x[i] = curr_x = curr_x + avg_vx * dt
        y[i] = curr_y = curr_y + avg_vy * dt
    return time,x,y

showscreen()
clearscreen()
radius,velocity = 150,10
for num in [100,200,500]:
    time_values,x,y = circular_motion2(radius,velocity,num)
    end_radius = np.sqrt(x[num-1]**2 + y[num-1]**2)   # Python 2 requires index num-1 instead of -1
    print('num =',num,', last radius / first radius =',round(end_radius/radius,5))
time_values,x,y = circular_motion2(radius,velocity,num=500)    
plot_xy(x,y,'red')      # need Python 2 to use turtle

Let's redo these calculations assuming constant acceleration toward the center.

In [ ]:
def circular_motion3(velocity,acceleration,num=500, circuits=3):
    radius = velocity**2 / acceleration
    agg_time =  circuits * (2 * np.pi * radius) / velocity
    time = np.linspace(0,agg_time,num+1)
    x,y = np.zeros(num+1),np.zeros(num+1)
    dt = agg_time / num
    x[0] = curr_x = radius
    curr_y = vx = 0
    vy = velocity
    for i in range(1,num+1):
        radius = np.sqrt(curr_x**2 + curr_y**2)
        avg_x = curr_x + 0.5 * vx * dt
        avg_y = curr_y + 0.5 * vy * dt
        avg_vx = vx - acceleration * (avg_x / radius) * dt
        avg_vy = vy - acceleration * (avg_y / radius) * dt
        x[i] = curr_x = curr_x + avg_vx * dt
        y[i] = curr_y = curr_y + avg_vy * dt
        vx = vx - acceleration * (avg_x / radius) * dt
        vy = vy - acceleration * (avg_y / radius) * dt  
    return time,x,y
        
velocity,acceleration = 10,0.5 
for num in [500,1000,10000,100000]:
    t,x,y = circular_motion3(velocity,acceleration,num)
    expansion = np.sqrt(x[-1]**2 + y[-1]**2) / np.sqrt(x[0]**2 + y[0]**2) 
    print('num =',num,', last radius / first radius =',round(expansion,5))

The previous orbits gradually spiral inward, 
as the method overstates the average contraction in velocity. 
The code below provides a much better approximation.

In [ ]:
def circular_motion4(velocity,acceleration,num=500, circuits=3): 
    radius = velocity**2 / acceleration
    agg_time = circuits * (2 * np.pi * radius) / velocity
    time = np.linspace(0,agg_time,num+1)
    x,y = np.zeros(num+1),np.zeros(num+1)
    dt = agg_time / num
    x[0] = curr_x = radius
    curr_y = vx = 0
    vy = velocity
    for i in range(1,num+1):
            radius = np.sqrt(curr_x**2 + curr_y**2)
            avg_vx = vx - 0.5 * acceleration * (curr_x / radius) * dt
            avg_vy = vy - 0.5 * acceleration * (curr_y / radius) * dt
            dx = avg_vx * dt
            dy = avg_vy * dt
            avg_x = curr_x + 0.5 * dx
            avg_y = curr_y + 0.5 * dy
            vx = vx - acceleration * (avg_x / radius) * dt
            vy = vy - acceleration * (avg_y / radius) * dt
            x[i] = curr_x = curr_x + dx
            y[i] = curr_y = curr_y + dy
    return time,x,y

showscreen()
clearscreen()
velocity,acceleration = 10,0.5 
for num in [100,200,500]:
    time_values,x,y = circular_motion4(velocity,acceleration,num)
    expansion = np.sqrt(x[num-1]**2 + y[num-1]**2) / np.sqrt(x[0]**2 + y[0]**2) 
    print('num =',num,', last radius / first radius =',round(expansion,5) )
plot_xy(x,y,'green')

The paths of *x* and *y* oscillate in unusually regular patterns. 
They are known as sine and cosine functions, which we will learn more about in Levels 5 and 7.

In [ ]:
showscreen()
clearscreen()
velocity,acceleration = 10,0.5 
time_values,x,y = circular_motion4(velocity,acceleration)
horizontal = np.linspace(-200,200,len(x))
plot_xy(horizontal,x,'blue')
plot_xy(horizontal,y,'red')

The gravitational acceleration induced by a large mass *M* at distance *r* is *GM/r*<sup>2</sup>, where *G* is the gravitational constant. 
Circular motion requires *v* = sqrt(*GM/r*). Let's rewrite the code to simulate any gravitational orbit, with circular motion as the default.

In [ ]:
def simulate_orbit(radius,GM,velocity=None,num=500,circuits=3):
    if velocity == None:   # assume circular orbit as default
        velocity = np.sqrt(GM / radius)
    period_approx = 2 * np.pi * radius / velocity # only a first approximation when orbit non-circular
    agg_time =  circuits * period_approx 
    time = np.linspace(0,agg_time,num+1)
    x,y = np.zeros(num+1),np.zeros(num+1)
    dt = agg_time / num
    x[0] = curr_x = radius
    curr_y = vx = 0
    vy = velocity
    for i in range(1,num+1):
        radius = np.sqrt(curr_x**2 + curr_y**2)
        gravity = GM / radius**2
        avg_vx = vx - 0.5 * gravity * (curr_x / radius) * dt
        avg_vy = vy - 0.5 * gravity * (curr_y / radius) * dt
        dx = avg_vx * dt
        dy = avg_vy * dt
        avg_x = curr_x + 0.5 * dx
        avg_y = curr_y + 0.5 * dy
        radius = np.sqrt(avg_x**2 + avg_y**2)
        gravity = GM / radius**2
        vx -= gravity * (avg_x / radius) * dt
        vy -= gravity * (avg_y / radius) * dt
        x[i] = curr_x = curr_x + dx
        y[i] = curr_y = curr_y + dy
    return time,x,y

showscreen()
clearscreen()radius,GM,velocity = 200,20000,10 
time_values,x,y = simulate_orbit(radius,GM,velocity)
plot_xy(x[::2],y[::2],'orange')

To better appreciate the impact of *r* on *v*, let's chart two orbits simultaneously. 

In [ ]:
showscreen()
clearscreen()
tu1 = Turtle()         # defines tu1 as object of Turtle class
tu1.penup()            # raises pen to avoid drawing
tu1.color('cyan')
tu1.shape('circle')
tu1.goto(100,50)
tu1.pendown()          # lowers pen to start drawing
tu1.begin_fill()
tu1.circle(75)
# create second turtle tu2 that draws pink circle
tu2 = Turtle()
tu2.penup()
tu2.color('pink')
tu2.shape('circle')
tu2.goto(-100,-100)
tu2.pendown()
tu2.begin_fill()
tu2.circle(100)
tu2.end_fill()   # fills the figure drawn by tu2
tu2.penup();
tu2.home()
tu2.pendown()
tu1.end_fill()   # fills the figure drawn by tu1

To apply this, we need to align the times of different simulations.

In [ ]:
showscreen()
clearscreen()
tu1 = Turtle(); tu1.shape('circle'); tu1.color('blue')
tu2 = Turtle(); tu2.shape('circle'); tu2.color('red')
radius,GM = 100,20000
time,x,y = simulate_orbit(radius,GM)
time2,x2,y2 = simulate_orbit(2*radius,GM)
tu2.penup(); tu2.goto(x2[0],y2[0]); tu2.pendown()
tu1.penup(); tu1.goto(x[0],y[0]); tu1.pendown()
k = 1
for i in range(1,len(time)):
    tu1.goto(x[i],y[i])
    while time2[k] <= time[i]:  # this approximately aligns the times 
        k+=1                    
    tu2.goto(x2[k-1],y2[k-1])

Now for something more exotic.  What happens if *GM* differs from *rv*<sup>2</sup>? 
The paths of *r*, *x*, and *y* can look very strange.

In [ ]:
showscreen()
clearscreen()
radius,GM,velocity = 200,30000,7
time,x,y = simulate_orbit(radius,GM,velocity,num=1000,circuits=1) 
distance_from_center = np.sqrt(x**2 + y**2)
horizontal = np.linspace(-200,100,len(time))
plot_xy(horizontal[::2],distance_from_center[::2])
t.write('distance from center')
plot_xy(horizontal[::2],x[::2],'green')
t.write('x coordinate')
plot_xy(horizontal[::2],y[::2],'red')
t.write('y coordinate')

However, the paths of the joint (*x,y*) are familiar. 
Are they true ellipses, hyperbolas, or something else? We'll explore this in Level 7.

In [ ]:
showscreen()
clearscreen()
radius,GM,velocity = 160,20000,10
time_values,x,y = simulate_orbit(radius,GM,velocity,circuits=1)
plot_xy(x[::2],y[::2],'orange')
time_values,x,y = simulate_orbit(radius,16000,velocity)
plot_xy(x[::2],y[::2],'purple')
time_values,x,y = simulate_orbit(radius,GM,7,circuits=1)
plot_xy(x[::2],y[::2],'green')

Let's form dictionaries for the periods of the planets and their relative distances from the Sun.
We combine *GM* = *r*<sup>2</sup> and *vT* = 2*r*$\pi$ to show that *r*<sup>3</sup> scales with *T(<sup>2</sup> for objects orbiting the same mass. 
(This holds for non-circular orbits too when *r* is defined appropriately.)

In [ ]:
planet_period = {'Mercury':0.24,'Venus':0.63,'Earth':1.0,'Mars':1.88,
                    'Jupiter':11.9,'Saturn':29.5,'Uranus':84,'Neptune':165}
planet_period2 = dict(Mercury=0.24,Venus=0.63,Earth=1.0,Mars=1.88,
                        Jupiter=11.9,Saturn=29.5,Uranus=84,Neptune=165) # alternative method
print('Does planet_period2 match planet_period1?',planet_period2==planet_period)
planet_distance = {key: round(value**(2/3),2) for key,value in planet_period.items()}
print('\nplanet_distance =',planet_distance)
planet_both = planet_period
for key,value in planet_both.items():
    planet_both[key] = [value, round(value**(2/3),2)]
print('\nplanet_both =',planet_both)